# Exercise: Deep Learning and Modern Forecasting

A streaming service needs to forecast monthly subscriber counts for the next year. Growth follows a classic S-curve — explosive early on, now decelerating. There's seasonality (fall signups, post-holiday churn) and structural breaks (a price hike in February 2023).

ARIMA struggles with nonlinear dynamics. N-BEATS learns its own features. Chronos-2 is a foundation model that recognizes the S-curve pattern zero-shot. Both produce probabilistic forecasts.

## Data

`../data/subscribers.csv` — monthly active subscribers from January 2020 through December 2024

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from darts import TimeSeries
from darts.models import NBEATSModel, ARIMA as DartsARIMA
from darts.metrics import mae, rmse, mape
from darts.utils.likelihood_models import QuantileRegression

HORIZON = 12

## Load and Explore Data

In [ ]:
df = pd.read_csv("../data/subscribers.csv", parse_dates=["date"], index_col="date")
df = df.asfreq("MS")
subscribers = df["subscribers"]

print(f"Data: {len(subscribers)} months, {subscribers.index[0].date()} to {subscribers.index[-1].date()}")
print(f"Range: {subscribers.min():,.0f} to {subscribers.max():,.0f}")

subscribers.plot(figsize=(14, 5), color="black", title="Monthly Active Subscribers")
plt.ylabel("Subscribers")
plt.tight_layout()
plt.show()

## Prepare Data

In [ ]:
# Prepare Darts TimeSeries and train/test split
ts = TimeSeries.from_series(subscribers)
train, test = ts.split_after(pd.Timestamp("2023-12-01"))
print(f"Train: {len(train)} months, Test: {len(test)} months")

## Your Task

Implement the four functions below:

1. **`train_nbeats`** — train an N-BEATS model with quantile regression
2. **`run_chronos_zero_shot`** — run Chronos-2 zero-shot forecast (with ARIMA fallback)
3. **`backtest_model`** — run historical forecasts and compute error metrics
4. **`build_comparison_table`** — create a formatted comparison DataFrame

In [ ]:
def train_nbeats(train, input_length=24, output_length=12, epochs=50):
    """Train an N-BEATS model with quantile regression. Return the fitted model."""
    # TODO: implement
    raise NotImplementedError


def run_chronos_zero_shot(train, horizon=12):
    """Run Chronos-2 zero-shot forecast. Fall back to ARIMA if unavailable."""
    # TODO: implement
    raise NotImplementedError


def backtest_model(model, series, start=0.75):
    """Run historical forecasts and return dict with mae, rmse, mape."""
    # TODO: implement
    raise NotImplementedError


def build_comparison_table(models):
    """Build a DataFrame comparing model metrics. `models` is {name: {mae, rmse, mape}}."""
    # TODO: implement
    raise NotImplementedError

## Train and Forecast

In [ ]:
# Train models
print("Training N-BEATS...")
nbeats = train_nbeats(train, epochs=50)
nbeats_fc = nbeats.predict(HORIZON, num_samples=100)

print("Running Chronos-2 zero-shot...")
chronos_fc = run_chronos_zero_shot(train, horizon=HORIZON)

print("Fitting ARIMA baseline...")
arima = DartsARIMA(p=2, d=1, q=1)
arima.fit(train)
arima_fc = arima.predict(HORIZON)

forecasts = {
    "ARIMA(2,1,1)": arima_fc,
    "N-BEATS": nbeats_fc,
    "Chronos-2": chronos_fc,
}

## Visualize Forecasts

In [ ]:
# Plot all forecasts
fig, ax = plt.subplots(figsize=(14, 7))
TimeSeries.from_series(subscribers).plot(ax=ax, label="Actual", color="black", linewidth=1.5)
colors = ["tab:blue", "tab:orange", "tab:green"]
for i, (name, fc) in enumerate(forecasts.items()):
    if fc.n_samples > 1:
        fc.plot(ax=ax, label=name, color=colors[i % len(colors)], low_quantile=0.05, high_quantile=0.95)
    else:
        fc.plot(ax=ax, label=name, color=colors[i % len(colors)])
ax.set_ylabel("Subscribers")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

print("\n=== Forecast Summary ===")
for name, fc in forecasts.items():
    end_val = fc.values().flatten()[-1]
    print(f"  {name}: {end_val:,.0f} subscribers at month 12")

## Backtest

In [ ]:
# Backtest
full_series = TimeSeries.from_series(subscribers)
metrics = {}
try:
    metrics["N-BEATS"] = backtest_model(nbeats, full_series)
except Exception:
    pass
metrics["ARIMA(2,1,1)"] = backtest_model(arima, full_series)

if metrics:
    table = build_comparison_table(metrics)
    print("\n=== Backtest Metrics ===")
    print(table)